# Marketing Analytics: CausalImpact & Robyn on Snowflake

An end-to-end marketing analytics workflow demonstrating how to run
R-native packages -- Google's **CausalImpact** and Meta's **Robyn** --
directly in Snowflake Workspace Notebooks, with data in Snowflake,
Feature Store pipelines, and Model Registry governance.

**What you'll do:**
1. Explore marketing data already in Snowflake
2. Build reusable Feature Store pipelines (aggregation, PIVOT, OBJECT_AGG)
3. Run CausalImpact to measure a campaign's causal effect
4. Run Robyn for Marketing Mix Modeling
5. Register models in the Snowflake Model Registry

**Database layout** (created by the setup notebook):

| Schema | Contents |
|---|---|
| `RAW_DATA` | Source tables -- campaigns, ad spend, revenue, holidays |
| `FEATURES` | Feature Store -- entity, feature views |
| `TRAINING` | Training datasets |
| `MODELS` | Model Registry -- versioned models, metrics |

**Prerequisites:** Run `workspace_marketing_setup.ipynb` first to install the
R environment, create the database/schemas, and generate source data.

**Sections:**
1. Setup and Connect
2. Exploratory Data Analysis
3. Feature Store
4. CausalImpact: Measuring Campaign Effect
5. Robyn: Marketing Mix Modeling
6. Model Registry Overview
7. Cleanup

## 1. Setup and Connect

`setup_notebook()` bootstraps the R environment inside the Workspace
container. Approximate timings on a Medium warehouse:

| Component | Source | Time | What it does |
|-----------|--------|------|--------------|
| **R 4.5 + rpy2** | conda-forge | ~25s | R runtime and Python-R bridge |
| **CausalImpact + tidyverse** | conda-forge | (bundled above) | Bayesian causal inference (Google), dplyr/ggplot2/dbplyr |
| **Robyn + nevergrad** | CRAN + pip | ~108s | Marketing Mix Modeling (Meta) + optimizer -- largest step |
| **snowflakeR** | GitHub tarball | ~6s | Feature Store, Model Registry, Datasets, query helpers |
| **RSnowflake** | GitHub tarball | ~5s | DBI-compliant Snowflake driver (REST API, SPCS OAuth) |
| **ML import pre-warm** | snowflake.ml | ~8s | Pre-loads Feature Store / Registry so first calls are fast |
| **Total** | | **~2m 50s** | One-time per session; subsequent cells start instantly |

After setup, we load R packages, connect to Snowflake, and create
DBI/dbplyr table references for downstream use.

In [ ]:
from sfnb_setup import setup_notebook
setup_notebook(config="snowflaker_marketing_config.yaml")

In [ ]:
%%R
library(snowflakeR)
library(CausalImpact)
library(tidyverse)
library(jsonlite)
library(scales)
rcat("snowflakeR", as.character(packageVersion("snowflakeR")), "loaded")
rcat("CausalImpact", as.character(packageVersion("CausalImpact")), "loaded")

### Connect and Configure Database Context

In [ ]:
%%R
# Snowpark session (powers all sfr_* calls: query, Feature Store, Model Registry)
conn <- sfr_connect()

# Database for demo
ma_db       <- "MARKETING_ANALYTICS_ML"
# Schemas for demo
ma_source   <- "RAW_DATA" # source tables
ma_features <- "FEATURES" # Feature Store
ma_training <- "TRAINING" # Training datasets
ma_registry <- "MODELS"   # Model Registry

# Fully qualified table references (DATABASE.SCHEMA.TABLE)
tbl_spend    <- sfr_fqn(conn, "DAILY_AD_SPEND",      database = ma_db, schema = ma_source)
tbl_revenue  <- sfr_fqn(conn, "DAILY_MARKET_REVENUE", database = ma_db, schema = ma_source)
tbl_holidays <- sfr_fqn(conn, "HOLIDAY_CALENDAR",    database = ma_db, schema = ma_source)

n <- sfr_query(conn, paste("SELECT COUNT(*) AS n FROM", tbl_spend))
rcat(sprintf("Connected. %s ad spend records in %s.",
             format(n$N, big.mark = ","), tbl_spend))

# DBI connection for dbplyr (reuses Snowpark auth, no second login)
dbi_con      <- sfr_dbi_connection(conn)
spend_tbl    <- tbl(dbi_con, I(tbl_spend))
revenue_tbl  <- tbl(dbi_con, I(tbl_revenue))
holidays_tbl <- tbl(dbi_con, I(tbl_holidays))

conn

## 2. Exploratory Data Analysis

Quick look at the marketing data before building models.

### Ad Spend by Channel Over Time

In [ ]:
%%R -w 900 -h 400
spend_weekly <- sfr_query(conn, paste(
  "SELECT DATE_TRUNC('week', DATE)::DATE AS WEEK_DATE,",
  "  CHANNEL, SUM(SPEND) AS TOTAL_SPEND",
  "FROM", tbl_spend,
  "GROUP BY 1, 2 ORDER BY 1, 2"
))
spend_weekly$WEEK_DATE <- as.Date(spend_weekly$WEEK_DATE)

suppressWarnings(print(
  ggplot(spend_weekly, aes(x = WEEK_DATE, y = TOTAL_SPEND, color = CHANNEL)) +
    geom_line(linewidth = 0.6) +
    scale_y_continuous(labels = comma) +
    labs(title = "Weekly Ad Spend by Channel",
         x = "Week", y = "Total Weekly Spend ($)", color = "Channel") +
    theme_minimal(base_size = 13) +
    theme(plot.title = element_text(face = "bold"), legend.position = "top")
))

TV spend (blue) shows a pronounced spike during Q3 2025 -- this is the Montreal TV campaign we will measure with CausalImpact. Facebook (green) has higher baseline spend but declines in late 2025. Search (red) is steady.

### Revenue Trends by Market

The red dashed line marks the start of the Montreal TV campaign.
Look for a visible uplift in Montreal's revenue after July 2025.

In [ ]:
%%R -w 900 -h 400
revenue_daily <- sfr_query(conn, paste(
  "SELECT DATE, MARKET, REVENUE FROM", tbl_revenue,
  "ORDER BY DATE, MARKET"
))
revenue_daily$DATE <- as.Date(revenue_daily$DATE)

suppressWarnings(print(
  ggplot(revenue_daily, aes(x = DATE, y = REVENUE, color = MARKET)) +
    geom_line(alpha = 0.2) +
    geom_smooth(method = "loess", span = 0.1, se = FALSE) +
    geom_vline(xintercept = as.Date("2025-07-01"),
               linetype = "dashed", color = "red") +
    annotate("text", x = as.Date("2025-07-15"),
             y = max(revenue_daily$REVENUE) * 0.95,
             label = "Campaign Start", hjust = 0, color = "red") +
    scale_y_continuous(labels = comma) +
    labs(title = "Daily Revenue by Market",
         subtitle = "Montreal TV campaign starts July 2025",
         x = "Date", y = "Daily Revenue ($)", color = "Market") +
    theme_minimal(base_size = 13) +
    theme(plot.title = element_text(face = "bold"), legend.position = "top")
))

Montreal revenue (red) rises sharply after the campaign start in July 2025, temporarily overtaking Toronto. The control markets (Toronto, Vancouver) follow their usual seasonal pattern, confirming the uplift is Montreal-specific.

### Montreal vs Control Markets (Monthly)

Smoothed monthly view makes the intervention effect clearer.
Toronto and Vancouver serve as control markets (Q3 2025).

In [ ]:
%%R -w 900 -h 400
monthly_rev <- revenue_daily %>%
  mutate(month = floor_date(DATE, "month")) %>%
  group_by(month, MARKET) %>%
  summarise(avg_revenue = mean(REVENUE), .groups = "drop")

suppressWarnings(print(
  ggplot(monthly_rev, aes(x = month, y = avg_revenue, color = MARKET)) +
    geom_line(linewidth = 1) +
    geom_vline(xintercept = as.Date("2025-07-01"),
               linetype = "dashed", color = "red") +
    scale_y_continuous(labels = comma) +
    labs(title = "Monthly Average Revenue: Montreal vs Control Markets",
         subtitle = "Montreal shows uplift during Q3 2025 campaign period",
         x = "Month", y = "Average Daily Revenue ($)", color = "Market") +
    theme_minimal(base_size = 13) +
    theme(plot.title = element_text(face = "bold"), legend.position = "top")
))

Smoothing to monthly averages makes the campaign effect unmistakable. Montreal (red) converges with Toronto around the Q3 2025 intervention, then falls back as the campaign ends. Vancouver (blue) tracks its own seasonal baseline throughout -- a clean control market.

## 3. Feature Store

Define reusable feature pipelines that transform raw marketing data
into model-ready formats. Each Feature View encapsulates a specific
transformation -- Snowflake handles execution and refresh.

| Feature View | Transformation | Snowflake technique | Feeds |
|---|---|---|---|
| `WEEKLY_CHANNEL_SPEND_FV` | Daily campaign spend to weekly channel totals | `GROUP BY` week + conditional aggregation | Robyn |
| `WEEKLY_REVENUE_FV` | Daily market revenue to weekly national totals | `GROUP BY` week + `SUM` | Robyn |
| `DAILY_MARKET_REVENUE_FV` | Daily revenue packed as JSON object by market | `OBJECT_AGG` | Base (generic) |
| `CI_MARKET_COLUMNS_FV` | Revenue pivoted to one column per market | `CASE WHEN` pivot | CausalImpact |

In [ ]:
%%R
# Create, or attach to existing feature-store
fs <- sfr_feature_store(conn,
  database  = ma_db,
  schema    = ma_features,
  warehouse = conn$warehouse,
  create    = TRUE # Create feature-store if it doesn't exist
)
rcat("Feature Store initialised.")


### Feature Store Entities

Entities define the **join keys** that align Feature Views when retrieving
training data. In a customer-level model you would use for example `CUSTOMER_ID`; here
both models are **time-series**, so the entity keys represent the time grains:

| Entity | Join Key | Used By |
|--------|----------|---------|
| `WEEKLY_GRAIN` | `WEEK_DATE` | Robyn (weekly spend + revenue) |
| `DAILY_GRAIN` | `DATE` | CausalImpact (daily market revenue) |

In [ ]:
%%R
weekly_grain <- sfr_create_entity(
  fs,
  name      = "WEEKLY_GRAIN",
  join_keys = "WEEK_DATE",
  desc      = "Weekly time grain for Robyn MMM features"
)

daily_grain <- sfr_create_entity(
  fs,
  name      = "DAILY_GRAIN",
  join_keys = "DATE",
  desc      = "Daily time grain for CausalImpact market features"
)

rcat("Entities registered.")
print(fs)


### Feature View 1: Weekly Channel Spend (dplyr/dbplyr pipeline)

Aggregates daily campaign-level ad spend into weekly totals per channel,
pivoted to wide format. This is the core data transformation for Robyn --
it needs one column per media variable.

We build the feature engineering query as a **dplyr/dbplyr pipeline** in R,
then render it to a **SQL string** with `dbplyr::sql_render()`. The SQL
is passed to `sfr_create_feature_view()`. This lets analysts author feature
logic in familiar R, while the pipeline executes entirely in Snowflake.

Uses a Dynamic Table (`refresh_freq = '1 hour'`) so features refresh
automatically as new spend data arrives.

Feature Views can also be created from Python using the `snowflake.ml.feature_store`
package. Regardless of whether a Feature View is created in R or Python, it is a
Snowflake-native object -- visible and usable from both languages.

In [ ]:
%%R
spend_query <- spend_tbl |>
  mutate(WEEK_DATE = as.Date(floor_date(DATE, "week"))) |>
  group_by(WEEK_DATE) |>
  summarise(
    TV_S            = sum(ifelse(CHANNEL == "TV",       SPEND, 0), na.rm = TRUE),
    FACEBOOK_S      = sum(ifelse(CHANNEL == "FACEBOOK", SPEND, 0), na.rm = TRUE),
    FACEBOOK_I      = sum(ifelse(CHANNEL == "FACEBOOK", IMPRESSIONS, 0), na.rm = TRUE),
    SEARCH_S        = sum(ifelse(CHANNEL == "SEARCH",   SPEND, 0), na.rm = TRUE),
    SEARCH_CLICKS_P = sum(ifelse(CHANNEL == "SEARCH",   CLICKS, 0), na.rm = TRUE),
    .groups = "drop"
  )

spend_sql <- dbplyr::sql_render(spend_query)
rcat("dplyr pipeline rendered to SQL:")
rcat(spend_sql)

In [ ]:
%%R
fv_channel_spend <- sfr_create_feature_view(
  fs,
  name         = "WEEKLY_CHANNEL_SPEND_FV",
  version      = "v1",
  entities     = weekly_grain,
  features     = spend_sql,
  refresh_freq = "1 hour",
  desc         = "Weekly channel spend pivot (Dynamic Table, from dplyr pipeline)",
  overwrite    = TRUE
)

rcat("Registered WEEKLY_CHANNEL_SPEND_FV (using SQL rendered from dplyr)")

### Feature View 2: Weekly Revenue (inline dplyr)

Aggregates daily market revenue into weekly national totals.
This provides the dependent variable (revenue) and conversions
for the Robyn model.

Here we pass the **dplyr/dbplyr lazy table directly** to
`sfr_create_feature_view()` -- contrast with the SQL string
approach used for FV1. Either way the pipeline runs entirely
in Snowflake.

In [ ]:
%%R
fv_weekly_revenue <- sfr_create_feature_view(
  fs,
  name     = "WEEKLY_REVENUE_FV",
  version  = "v1",
  entities = weekly_grain,
  features = revenue_tbl |>
    mutate(WEEK_DATE = as.Date(floor_date(DATE, "week"))) |>
    group_by(WEEK_DATE) |>
    summarise(
      REVENUE     = sum(REVENUE, na.rm = TRUE),
      CONVERSIONS = sum(CONVERSIONS, na.rm = TRUE),
      .groups = "drop"
    ),
  refresh_freq = "1 hour",
  desc      = "Weekly national revenue totals (Dynamic Table, inline dplyr)",
  overwrite = TRUE
)
rcat("Registered WEEKLY_REVENUE_FV (inline dplyr pipeline)")

### Feature View 3: Daily Market Revenue (OBJECT_AGG) -- Base View

Packs per-market daily revenue into a single Snowflake `OBJECT` column
using `OBJECT_AGG`. Market names become keys in the JSON object.

This is **deliberately generic** -- the Feature View never hardcodes
market names. New markets appear automatically as keys. It serves as the
flexible base layer; model-specific views (below) derive from the same
source table and expand only the markets they need.

In [ ]:
%%R
fv_market_revenue <- sfr_create_feature_view(
  fs,
  name     = "DAILY_MARKET_REVENUE_FV",
  version  = "v1",
  entities = daily_grain,
  features = paste(
    "SELECT DATE,",
    "  OBJECT_AGG(MARKET, REVENUE::VARIANT) AS MARKET_REVENUE",
    "FROM", tbl_revenue,
    "GROUP BY DATE"
  ),
  desc      = "Daily revenue as OBJECT keyed by market (OBJECT_AGG)",
  overwrite = TRUE
)
rcat("Registered DAILY_MARKET_REVENUE_FV")

In [ ]:
%%R
rcat("Feature Views registered:")
sfr_list_feature_views(fs)

### Feature View 4: CausalImpact Market Columns (derived from FV3)

A model-specific Feature View that extracts the three markets used by
CausalImpact (Montreal = treatment, Toronto and Vancouver = controls)
from the OBJECT_AGG Dynamic Table created by Feature View 3.

This creates a proper **feature engineering pipeline**: the raw source
table is aggregated into a JSON object by FV3's Dynamic Table, and this
view then extracts the required keys as typed columns. Changes to the
upstream FV3 automatically flow through.

The Dataset generated from this Feature View has columns that match the
model's `input_cols` directly, enabling full ML Lineage:
Source Table -> FV3 (OBJECT_AGG DT) -> FV4 (columnar) -> Dataset -> Model.

In [ ]:
%%R
treatment <- "MONTREAL"
controls  <- c("TORONTO", "VANCOUVER")
markets   <- c(treatment, controls)

# Reference the Dynamic Table backing the OBJECT_AGG Feature View
dt_ref <- sprintf('%s.FEATURES."DAILY_MARKET_REVENUE_FV$v1"', ma_db)

# Extract JSON keys from the OBJECT column as typed columns
extract_sql <- paste(
  "SELECT DATE,",
  paste(sprintf(
    "MARKET_REVENUE:%s::DOUBLE AS %s", markets, markets
  ), collapse = ",\n  "),
  "FROM", dt_ref
)

fv_ci_markets <- sfr_create_feature_view(
  fs,
  name     = "CI_MARKET_COLUMNS_FV",
  version  = "v1",
  entities = daily_grain,
  features = extract_sql,
  desc     = "Market revenue columns extracted from OBJECT_AGG DT (Montreal, Toronto, Vancouver)",
  overwrite = TRUE
)
rcat("Registered CI_MARKET_COLUMNS_FV")

## 4. CausalImpact: Measuring Campaign Effect

Google's CausalImpact uses Bayesian structural time series to estimate
the causal effect of an intervention. We ask: *"Did the Montreal Q3 2025 TV
campaign actually drive incremental revenue, or was it a normal
seasonal pattern?"*

The model uses Toronto and Vancouver as control markets to build a
counterfactual -- what Montreal's revenue *would have been* without
the campaign.

In [ ]:
%%R
# Define Spine of dates to be used for analysis
date_spine <- revenue_tbl |>
  distinct(DATE) |>
  arrange(DATE)

# Generate a versioned Dataset from the columnar Feature View.
# The Dataset columns (MONTREAL, TORONTO, VANCOUVER) match the model's
# input_cols, enabling full ML Lineage (Feature View -> Dataset -> Model).
market_data <- sfr_generate_dataset(
  fs,
  name     = "CI_DAILY_MARKET_REVENUE",
  spine    = date_spine,
  features = list(fv_ci_markets),
  version  = "v1"
)

rcat(sprintf("Dataset %s (v%s): %d rows x %d cols",
             attr(market_data, "dataset_name"),
             attr(market_data, "dataset_version"),
             nrow(market_data), ncol(market_data)))
head(market_data, 3)

### Fit CausalImpact Model

Define the pre-intervention period (before the campaign) and the
post-intervention period (during the campaign). CausalImpact learns
the relationship between treatment and control markets in the pre-period,
then predicts the counterfactual in the post-period.

In [ ]:
%%R
market_data$DATE <- as.Date(market_data$DATE)
market_data <- market_data[order(market_data$DATE), ]

intervention_date <- as.Date("2025-07-01")
pre_end    <- max(which(market_data$DATE < intervention_date))
post_start <- min(which(market_data$DATE >= intervention_date))

pre.period  <- c(1, pre_end)
post.period <- c(post_start, nrow(market_data))

rcat(sprintf("Pre-period:  rows %d-%d (%s to %s)",
             pre.period[1], pre.period[2],
             market_data$DATE[pre.period[1]], market_data$DATE[pre.period[2]]))
rcat(sprintf("Post-period: rows %d-%d (%s to %s)",
             post.period[1], post.period[2],
             market_data$DATE[post.period[1]], market_data$DATE[post.period[2]]))

data_matrix <- as.matrix(market_data[, c(treatment, controls)])
impact <- CausalImpact(data_matrix, pre.period, post.period)
rcat("CausalImpact model fitted.")

### Results

Three panels:
1. **Original** -- actual Montreal revenue vs counterfactual (predicted without campaign)
2. **Pointwise** -- day-by-day causal effect (actual minus counterfactual)
3. **Cumulative** -- running total of incremental revenue from the campaign

In [ ]:
%%R -w 900 -h 600
plot(impact)

In [ ]:
%%R
summary(impact)

In [ ]:
%%R
summary(impact, "report")

### Register CausalImpact Model in Model Registry

`sfr_log_model()` serialises the R model to RDS, wraps it in a Snowflake
ML `CustomModel`, and registers it in the Model Registry. Key arguments:

| Argument | Purpose |
|----------|---------|
| `model` | The fitted R object to serialise (here, the raw bsts model inside the CausalImpact result) |
| `predict_body` | Custom R code executed at inference time inside the container (see below) |
| `predict_pkgs` | R packages loaded before the predict code runs |
| `conda_deps` | conda-forge packages installed in the SPCS container |
| `input_cols` / `output_cols` | Column schemas so Snowflake can validate inputs and outputs |

**Why `predict_body`?** The default `predict(model, newdata)` pattern works
for most models, but `bsts::predict()` has a non-standard signature -- it
requires `horizon` and `newdata` separately, and returns a list with `$mean`
and `$interval` that must be reshaped into a data.frame. `predict_body`
lets you define that custom inference logic as a template.

The template uses `{{MODEL}}`, `{{INPUT}}`, and `{{UID}}` placeholders that
are substituted at runtime. The code must produce a `result_{{UID}}`
data.frame matching `output_cols`.

In [ ]:
%%R
# Create (or open) the model registry object
reg <- sfr_model_registry(conn,
  database = ma_db,
  schema   = ma_registry
)
rcat(sprintf("Model Registry: %s.%s", ma_db, ma_registry))


In [ ]:
%%R
# Define the inference logic as a readable R function.
# bsts::predict() requires horizon + newdata (non-standard), and its
# output ($mean, $interval) must be reshaped into a data.frame.
ci_predict <- function(model, input) {
  nd     <- as.matrix(input[, c("TORONTO", "VANCOUVER"), drop = FALSE])
  pred   <- predict(model, horizon = nrow(nd), newdata = nd)
  result <- data.frame(
    period         = seq_len(nrow(nd)),
    point_forecast = as.numeric(pred$mean),
    lower_95       = as.numeric(pred$interval[, 1]),
    upper_95       = as.numeric(pred$interval[, 2])
  )
}

# Convert the function body to a predict_body template.
# sfr_predict_body() replaces formal args with {{MODEL}}, {{INPUT}},
# and {{UID}} placeholders for execution inside the SPCS container.
ci_body <- sfr_predict_body(ci_predict)

mv_ci <- sfr_log_model(
  reg,
  model             = impact$model$bsts.model,
  model_name        = "CAUSAL_IMPACT_CAMPAIGN",
  predict_pkgs      = c("bsts"),
  predict_body      = ci_body,
  conda_deps        = c("r-bsts", "r-boom", "r-zoo"),
  input_cols        = list(TORONTO = "double", VANCOUVER = "double"),
  output_cols       = list(
    period = "integer", point_forecast = "double",
    lower_95 = "double", upper_95 = "double"
  ),
  training_dataset  = market_data,
  comment = "CausalImpact bsts model - Montreal Q3 2025 TV campaign"
)

rcat(sprintf("Registered: %s version %s",
             mv_ci$model_name, mv_ci$version_name))

### CausalImpact Inference

The CausalImpact bsts model is now registered in the Model Registry. Below we
demonstrate two local inference paths:

1. **R** -- predict directly from the in-memory bsts model (the analyst workflow)
2. **Python** -- retrieve the model from the registry via `ModelVersion.load()` and predict (cross-language roundtrip)

A final reference section shows how to deploy the model as a production SPCS
service for R, Python, and SQL access.

In [ ]:
%%R
bsts_model <- impact$model$bsts.model

# bsts needs future values for the regression controls (TORONTO, VANCOUVER).
# Use the last 30 days of observed control data as a realistic forecast input.
newdata <- tail(data_matrix[, controls, drop = FALSE], 30)
pred <- predict(bsts_model, horizon = nrow(newdata), newdata = newdata)

forecast_r <- data.frame(
  period         = seq_len(nrow(newdata)),
  point_forecast = as.numeric(pred$mean),
  lower_95       = as.numeric(pred$interval[, 1]),
  upper_95       = as.numeric(pred$interval[, 2])
)

rcat("30-day forecast from in-memory bsts model:")
head(forecast_r, 10)

### Retrieve from Model Registry and Predict (Python)

Load the registered R model back from the Model Registry using the Python SDK.
`ModelVersion.load()` downloads the model artifacts (including the `.rds` file)
and reconstructs the `CustomModel` wrapper. Because the Workspace notebook has
rpy2, R, and bsts installed, the wrapper can invoke R prediction under the hood.

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.snowpark.context import get_active_session
import rpy2.robjects as ro
import pandas as pd

session = get_active_session()
reg_py = Registry(session=session,
                  database_name="MARKETING_ANALYTICS_ML",
                  schema_name="MODELS")

# Use the version just registered by cell 37 (mv_ci), not m.default
# which may point to a stale version with the old predict_body.
version_name = str(ro.r('mv_ci$version_name')[0])
m = reg_py.get_model("CAUSAL_IMPACT_CAMPAIGN")
mv = m.version(version_name)
print(f"Loading model version: {version_name}")

model = mv.load(force=True)

# Build input_df with the control-market columns the predict_body expects
ctrl_r = ro.r('tail(data_matrix[, controls, drop = FALSE], 30)')
input_df = pd.DataFrame({
    "TORONTO": list(ctrl_r.rx(True, "TORONTO")),
    "VANCOUVER": list(ctrl_r.rx(True, "VANCOUVER")),
})
forecast_py = model.predict(input_df)
print("30-day forecast retrieved from Model Registry (Python):")
print(forecast_py.head(10))

### Production Serving via SPCS (Reference)

For production workloads the registered model can be deployed as an auto-scaling
REST service on Snowpark Container Services. All CausalImpact dependencies
(`bsts`, `Boom`, `zoo`) are on conda-forge, so the SPCS image build succeeds.
Below we use `snowflakeR` to deploy, but the Python `snowflake-ml` package
exposes identical functionality -- once a model is in the Registry it is
language-agnostic.

**Deploy the service (R):**

```r
sfr_create_compute_pool(conn, "MARKETING_POOL",
                        instance_family = "CPU_X64_M")
sfr_create_image_repo(conn, "MARKETING_ANALYTICS_ML.MODELS.MARKETING_IMAGES")

sfr_deploy_model(
  reg,
  model_name   = "CAUSAL_IMPACT_CAMPAIGN",
  version_name = mv_ci$version_name,
  service_name = "causal_impact_svc",
  compute_pool = "MARKETING_POOL",
  image_repo   = "MARKETING_ANALYTICS_ML.MODELS.MARKETING_IMAGES",
  force        = TRUE
)
sfr_wait_for_service(reg, "causal_impact_svc", timeout_min = 10)
```

**Score from R:**

```r
input_df <- data.frame(horizon = rep(1L, 30))
preds <- sfr_predict(reg, "CAUSAL_IMPACT_CAMPAIGN", input_df,
                     version_name = mv_ci$version_name,
                     service_name = "causal_impact_svc")
```

**Score from Python:**

```python
result = mv.run(
    session.create_dataframe(pd.DataFrame({"HORIZON": [1] * 30})),
    service_name="causal_impact_svc"
)
result.show()
```

**Score from SQL:**

```sql
SELECT t.*, MARKETING_ANALYTICS_ML.MODELS.CAUSAL_IMPACT_SVC!PREDICT(HORIZON) AS forecast
FROM (SELECT 1 AS HORIZON FROM TABLE(GENERATOR(ROWCOUNT => 30))) t
```

The SQL path means any Snowflake user, BI tool, or scheduled task can call the
model with no R or Python required.

## 5. Robyn: Marketing Mix Modeling

Meta's Robyn performs semi-automated Marketing Mix Modeling using ridge
regression with Adstock and saturation transformations. It uses Python's
`nevergrad` optimizer for multi-objective hyperparameter tuning.

In a Workspace Notebook, R runs inside Python via `rpy2`, so `nevergrad`
is already accessible -- no manual `reticulate` configuration is needed.
The `setup_notebook()` call at the top of this notebook installed both
Robyn (from CRAN) and `nevergrad` (via pip).

### Generate Training Dataset for Robyn

Build a weekly date spine using dplyr (evaluated as SQL by dbplyr) and
join it with both Robyn Feature Views via `sfr_generate_dataset()`. This
creates an immutable, versioned Snowflake ML Dataset that participates in
ML Lineage (Feature View -> Dataset -> Model). The Feature Store handles
the joins on `WEEK_DATE`.

In [ ]:
%%R
# Define Spine of weeks to be used for modelling
week_spine <- revenue_tbl |>
  mutate(WEEK_DATE = as.Date(floor_date(DATE, "week"))) |>
  distinct(WEEK_DATE)

# Generate a versioned Dataset from the Feature Store.
# The Dataset is immutable and participates in ML Lineage
# (Feature View -> Dataset -> Model).
robyn_training <- sfr_generate_dataset(
  fs,
  name     = "ROBYN_WEEKLY_TRAINING",
  spine    = week_spine,
  features = list(fv_channel_spend,
                  fv_weekly_revenue),
  version  = "v1"
)

rcat(sprintf("Dataset %s (v%s): %d rows x %d cols",
             attr(robyn_training, "dataset_name"),
             attr(robyn_training, "dataset_version"),
             nrow(robyn_training), ncol(robyn_training)))
head(as.data.frame(robyn_training), 5)

### Load Robyn and Prepare Training Data

Read the training data generated by Feature Store and rename columns
to match Robyn's expected naming convention. Also read the holiday
calendar from Snowflake.

In [ ]:
%%R
library(Robyn)

# Point cmdstanr at conda-installed cmdstan (avoids rstan recompilation)
if (requireNamespace("cmdstanr", quietly = TRUE)) {
  cmdstan_bin <- Sys.which("cmdstan")
  if (nzchar(cmdstan_bin)) {
    cmdstanr::set_cmdstan_path(dirname(dirname(cmdstan_bin)))
  }
}

# Read the training data and rename columns to match Robyn's expected naming convention
robyn_data <- robyn_training %>%
  rename(DATE = WEEK_DATE) %>%
  mutate(DATE = as.Date(DATE)) %>%
  rename_with(tolower) %>%
  arrange(date)

# Read the holiday data and rename columns to match Robyn's expected naming convention
holidays <- collect(holidays_tbl)
holidays <- holidays %>%
  rename_with(tolower) %>%
  mutate(ds = as.Date(ds), year = as.integer(format(ds, "%Y")))

# Print the data summary
rcat(sprintf("Robyn data: %d weeks x %d cols", nrow(robyn_data), ncol(robyn_data)))
rcat(sprintf("Holidays: %d entries", nrow(holidays)))
head(robyn_data)

### Model Specification

Define hyperparameter ranges for three channels using geometric adstock
(one decay parameter per channel). The search space is kept minimal
for demo speed.

> **Robyn Bug Workaround — Hyperparameter Naming**
>
> We discovered a bug in Robyn's `robyn_inputs()` (Use Case 1 code path in
> `R/inputs.R`) that causes `robyn_run()` to fail with:
>
> ```
> Error in names(hyper_list_all) <- `*vtmp*` :
>   'names' attribute [1] must be the same length as the vector [0]
> ```
>
> The bug is triggered when `paid_media_spends` and `paid_media_vars` use
> different column names (e.g. spend = `facebook_s`, exposure = `facebook_i`).
> Internally, `check_hyperparameters()` correctly renames the hyperparameters
> from spend-based to var-based names — but stores the result in a **local
> variable** and never writes it back to `InputCollect$hyperparameters`.
> When `robyn_run()` later calls `hyper_collector()`, it looks for var-based
> names that don't exist and crashes.
>
> **Workaround:** define hyperparameters using the `paid_media_vars` names
> directly (`facebook_i_*`, `search_clicks_p_*`) instead of the
> `paid_media_spends` names (`facebook_s_*`, `search_s_*`). This bypasses the
> broken renaming path entirely.

In [ ]:
%%R
# Hyperparameter names must use paid_media_vars names (not paid_media_spends)
# when spends and vars differ.  Robyn's robyn_inputs() has a bug where the
# internal spend-to-var renaming is not persisted to InputCollect.
hyperparameters <- list(
  tv_s_alphas = c(0.5, 3),
  tv_s_gammas = c(0.3, 1),
  tv_s_thetas = c(0.3, 0.8),
  facebook_i_alphas = c(0.5, 3),
  facebook_i_gammas = c(0.3, 1),
  facebook_i_thetas = c(0, 0.3),
  search_clicks_p_alphas = c(0.5, 3),
  search_clicks_p_gammas = c(0.3, 1),
  search_clicks_p_thetas = c(0, 0.3)
)

InputCollect <- robyn_inputs(
  dt_input = robyn_data,
  dt_holidays = holidays,
  date_var = "date",
  dep_var = "revenue",
  dep_var_type = "revenue",
  prophet_vars = c("trend", "season", "holiday"),
  prophet_country = "CA",
  paid_media_spends = c("tv_s", "facebook_s", "search_s"),
  paid_media_vars = c("tv_s", "facebook_i", "search_clicks_p"),
  window_start = "2024-01-01",
  window_end = "2026-02-28",
  adstock = "geometric",
  hyperparameters = hyperparameters
)

print(InputCollect)

### Train Model

Run the optimizer using all available cores. For a live demo we use
reduced iterations (500) and a single trial (~30 sec). Production runs
typically use 2000+ iterations and 5 trials for convergence.

In [ ]:
%%R
set.seed(42)
n_cores <- 1L
rcat(sprintf("Training with %d cores", n_cores))

OutputModels <- robyn_run(
  InputCollect = InputCollect,
  cores = n_cores,
  iterations = 500,
  trials = 1,
  ts_validation = FALSE,
  add_penalty_factor = FALSE
)

### View Results

Generate Pareto-optimal model outputs. The `export = FALSE` and
`plot_pareto = FALSE` flags are set for headless notebook execution.

In [ ]:
%%R
OutputCollect <- robyn_outputs(
  InputCollect, OutputModels,
  pareto_fronts = "auto",
  csv_out = "pareto",
  clusters = TRUE,
  export = FALSE,
  plot_pareto = FALSE
)

print(OutputCollect)

### Serving Architecture Discussion

Robyn and its dependency `lares` are currently **not on conda-forge**, which means
the model cannot be served directly via `sfr_deploy_model()` from the Model Registry.
Three production-ready alternatives:

**Option A: Extract the glmnet model (recommended for scoring)**
Register only the underlying ridge regression (`glmnet`) with manually
coded Adstock/saturation transforms. `glmnet` IS on conda-forge.

**Option B: Pre-compute response curves (recommended for planning)**
Export Robyn's response curves as a Snowflake table or Python model.
A simple SQL JOIN or interpolation UDF serves spend-to-response lookups
with no R dependencies at inference.

**Option C: Model Registry for governance + custom SPCS for serving**
Register the full Robyn model in the Model Registry via `sfr_log_model()`
for versioning, metrics, and lineage -- this works today (the conda-forge
check is a warning, not a blocker). However, `sfr_deploy_model()` cannot
auto-build the inference container because Robyn/lares aren't on conda-forge.
For live serving, build a custom SPCS container image with Robyn
pre-installed and load the model from the registry.

### Robyn Inference: Pre-computed Response Curves (Option B)

We implement **Option B** from the serving discussion above. The idea:

1. Use `robyn_response()` to evaluate each channel's spend-to-revenue curve
   at a grid of spend levels
2. Write the response curves to a **Snowflake table** — no R dependencies needed
3. At inference time, query the table with **pure SQL** for budget planning

This is the recommended approach for marketing planning because:
- **Fast** -- inference is pure Snowflake SQL (joins + scalar arithmetic on the
  pre-computed table), with no model evaluation or R/Python runtime needed
- **Accessible** -- any analyst can query it with standard SQL
- **Scalable** -- pairs naturally with `robyn_allocator()` for budget optimization

In [ ]:
%%R
# Pick the best model: Robyn selects one automatically (selectID),
# otherwise fall back to the first Pareto-optimal solution
best_model <- if (!is.null(OutputCollect$selectID)) {
  OutputCollect$selectID
} else {
  OutputCollect$allSolutions[1]
}
rcat(sprintf("Best Robyn model: %s", best_model))

# Channels to evaluate (must match paid_media_vars names, not spend names).
# robyn_response() requires date_range when metric_value is provided, and
# returns $sim_mean_response (not $response).
media_vars <- c("tv_s", "facebook_i", "search_clicks_p")
grid_size  <- 20  # number of spend levels per channel

# For each channel, evaluate the response curve at a grid of spend levels
# from 0 to 1.5x the historical maximum
response_data <- do.call(rbind, lapply(media_vars, function(mv) {
  hist_vals <- robyn_data[[mv]]
  max_val   <- max(hist_vals, na.rm = TRUE) * 1.5  # extend 50% beyond observed max
  val_grid  <- seq(max_val / grid_size, max_val, length.out = grid_size)

  # Evaluate Robyn's fitted response at each grid point
  responses <- sapply(val_grid, function(v) {
    r <- robyn_response(
      InputCollect  = InputCollect,
      OutputCollect = OutputCollect,
      select_model  = best_model,
      metric_name   = mv,
      metric_value  = v,
      date_range    = "last_1",
      quiet         = TRUE
    )
    r$sim_mean_response
  })

  # Marginal return = slope between adjacent grid points (dRevenue/dSpend)
  marginal <- c(NA_real_, diff(responses) / diff(val_grid))

  data.frame(
    CHANNEL           = mv,
    MEDIA_VALUE       = round(val_grid, 2),
    PREDICTED_REVENUE = round(responses, 2),
    MARGINAL_RETURN   = round(marginal, 4)
  )
}))

rcat(sprintf("Response curves: %d points across %d channels",
             nrow(response_data), length(media_vars)))
head(response_data, 10)

The chart below plots the **pre-computed response curves** for each media
channel. The x-axis is weekly media volume (spend, impressions, or clicks)
and the y-axis is the predicted revenue contribution. The characteristic
S-shaped curves show **diminishing returns** at higher spend -- the flattening
slope indicates saturation, where additional media investment yields
progressively less incremental revenue.

In [ ]:
%%R -w 900 -h 450
library(ggplot2)

channel_labels <- c(
  tv_s            = "TV (spend $)",
  facebook_i      = "Facebook (impressions)",
  search_clicks_p = "Search (clicks)"
)

ggplot(response_data, aes(x = MEDIA_VALUE, y = PREDICTED_REVENUE,
                          colour = CHANNEL)) +
  geom_line(linewidth = 1.2) +
  scale_colour_manual(values = c(tv_s = "#1b9e77", facebook_i = "#d95f02",
                                 search_clicks_p = "#7570b3"),
                      labels = channel_labels) +
  scale_x_continuous(labels = scales::comma) +
  scale_y_continuous(labels = scales::comma) +
  labs(title    = "Robyn Response Curves by Channel",
       subtitle = "Diminishing returns visible — saturation effects at higher media volume",
       x = "Weekly Media Volume (spend / impressions / clicks)",
       y = "Predicted Revenue Contribution ($)",
       colour = "Channel") +
  theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = "bold"), legend.position = "top")

In [ ]:
%%R
# Tag each row with the model ID and generation timestamp for lineage
response_data$MODEL_ID  <- best_model
response_data$GENERATED <- Sys.time()

# Write pre-computed response curves to Snowflake -- this is the table
# that downstream SQL queries join against for budget planning
curves_table <- sfr_fqn(conn, "ROBYN_RESPONSE_CURVES", database = ma_db, schema = ma_registry)
sfr_write_table(conn, response_data, table = curves_table, overwrite = TRUE)
rcat(sprintf("Written %d rows to %s", nrow(response_data), curves_table))

# Extract channel decomposition from Robyn's output:
# coefficient (ridge weight), total contribution, and % share of revenue
xda <- OutputCollect$xDecompAgg
decomp <- xda %>%
  filter(solID == best_model, rn %in% media_vars) %>%
  select(rn, coef, xDecompAgg, xDecompPerc)
names(decomp) <- c("CHANNEL", "COEFFICIENT", "TOTAL_CONTRIBUTION", "CONTRIBUTION_PCT")

# Write decomposition table -- useful for channel-level ROI reporting
decomp_table <- sfr_fqn(conn, "ROBYN_CHANNEL_DECOMP", database = ma_db, schema = ma_registry)
sfr_write_table(conn, decomp, table = decomp_table, overwrite = TRUE)
rcat(sprintf("Written %d rows to %s", nrow(decomp), decomp_table))
decomp

### SQL Inference: "What if we increase media volume by 20%?"

Now that the response curves live in a Snowflake table, any analyst can
answer budget planning questions with pure SQL — no R, no Python, no
model loading. The query finds the nearest pre-computed grid point for
each channel at both the current and proposed (+20%) media levels.

In [ ]:
%%R
curves_tbl <- tbl(dbi_con, I(sfr_fqn(conn, "ROBYN_RESPONSE_CURVES", database = ma_db, schema = ma_registry)))

avg_level <- curves_tbl %>%
  filter(MEDIA_VALUE > 0) %>%
  group_by(CHANNEL) %>%
  summarise(AVG_VALUE = mean(MEDIA_VALUE, na.rm = TRUE), .groups = "drop")

nearest_now <- curves_tbl %>%
  inner_join(avg_level, by = "CHANNEL") %>%
  mutate(DIST = abs(MEDIA_VALUE - AVG_VALUE)) %>%
  group_by(CHANNEL) %>%
  slice_min(DIST, n = 1) %>%
  ungroup() %>%
  select(CHANNEL, AVG_VALUE,
         REVENUE_NOW = PREDICTED_REVENUE, MR_NOW = MARGINAL_RETURN)

nearest_new <- curves_tbl %>%
  inner_join(avg_level, by = "CHANNEL") %>%
  mutate(DIST = abs(MEDIA_VALUE - AVG_VALUE * 1.2)) %>%
  group_by(CHANNEL) %>%
  slice_min(DIST, n = 1) %>%
  ungroup() %>%
  select(CHANNEL, REVENUE_NEW = PREDICTED_REVENUE, MR_NEW = MARGINAL_RETURN)

what_if <- nearest_now %>%
  inner_join(nearest_new, by = "CHANNEL") %>%
  transmute(
    CHANNEL,
    CURRENT_MEDIA_LEVEL     = round(AVG_VALUE),
    PROPOSED_LEVEL_20PCT_UP = round(AVG_VALUE * 1.2),
    REVENUE_AT_CURRENT      = round(REVENUE_NOW),
    REVENUE_AT_PROPOSED     = round(REVENUE_NEW),
    INCREMENTAL_REVENUE     = round(REVENUE_NEW - REVENUE_NOW),
    MARGINAL_RETURN_AT_PROPOSED = round(MR_NEW, 4)
  ) %>%
  arrange(CHANNEL) %>%
  collect()

# Show the SQL that dbplyr generated -- this query can run in any
# SQL client (Snowsight, BI dashboard) with no R or Python needed
what_if_sql <- dbplyr::sql_render(
  nearest_now %>%
    inner_join(nearest_new, by = "CHANNEL") %>%
    transmute(
      CHANNEL,
      CURRENT_MEDIA_LEVEL     = round(AVG_VALUE),
      PROPOSED_LEVEL_20PCT_UP = round(AVG_VALUE * 1.2),
      REVENUE_AT_CURRENT      = round(REVENUE_NOW),
      REVENUE_AT_PROPOSED     = round(REVENUE_NEW),
      INCREMENTAL_REVENUE     = round(REVENUE_NEW - REVENUE_NOW),
      MARGINAL_RETURN_AT_PROPOSED = round(MR_NEW, 4)
    ) %>%
    arrange(CHANNEL)
)
rcat("Generated SQL (copy into any SQL client or BI tool):")
rcat(what_if_sql)

rcat("Scenario: +20% media volume across all channels")
what_if

The dumbbell chart below visualises the revenue impact of the +20%
media volume scenario. Each channel shows the current revenue contribution
(orange) and the proposed level (green), with the incremental dollar gain
labelled. Channels with steeper response curves show larger gains;
saturated channels show smaller incremental returns.

In [ ]:
%%R -w 900 -h 400
channel_labels <- c(
  tv_s = "TV (spend)", facebook_i = "Facebook (impressions)",
  search_clicks_p = "Search (clicks)"
)

scenario <- do.call(rbind, lapply(media_vars, function(mv) {
  rd <- response_data[response_data$CHANNEL == mv, ]
  avg_val <- mean(rd$MEDIA_VALUE)
  idx_now <- which.min(abs(rd$MEDIA_VALUE - avg_val))
  idx_new <- which.min(abs(rd$MEDIA_VALUE - avg_val * 1.2))
  data.frame(
    CHANNEL = mv,
    LABEL = channel_labels[mv],
    CURRENT = rd$PREDICTED_REVENUE[idx_now],
    PROPOSED = rd$PREDICTED_REVENUE[idx_new],
    DELTA = rd$PREDICTED_REVENUE[idx_new] - rd$PREDICTED_REVENUE[idx_now]
  )
}))

ggplot(scenario) +
  geom_segment(aes(x = CURRENT, xend = PROPOSED, y = reorder(LABEL, DELTA),
                   yend = reorder(LABEL, DELTA)),
               linewidth = 2, colour = "grey80") +
  geom_point(aes(x = CURRENT, y = reorder(LABEL, DELTA)),
             size = 5, colour = "#d95f02") +
  geom_point(aes(x = PROPOSED, y = reorder(LABEL, DELTA)),
             size = 5, colour = "#1b9e77") +
  geom_text(aes(x = PROPOSED, y = reorder(LABEL, DELTA),
                label = sprintf("+$%s", format(round(DELTA), big.mark = ","))),
            hjust = -0.15, size = 4.5, fontface = "bold", colour = "#1b9e77") +
  scale_x_continuous(labels = scales::dollar_format(), expand = expansion(mult = c(0.05, 0.2))) +
  labs(title = "Revenue Impact: +20% Media Volume by Channel",
       subtitle = "Orange = current level  |  Green = +20% proposed",
       x = "Predicted Weekly Revenue Contribution",
       y = NULL) +
  theme_minimal(base_size = 14) +
  theme(plot.title = element_text(face = "bold"),
        panel.grid.major.y = element_blank())

### Key Takeaway

The dumbbell chart reveals diminishing returns at work across channels:

- **Search (clicks)** delivers the highest marginal return — a 20% increase in click volume produces meaningful incremental revenue, indicating headroom for growth.
- **TV (spend)** shows moderate uplift — still responsive to increased investment but approaching saturation.
- **Facebook (impressions)** is fully saturated — a 20% increase in impressions yields essentially zero additional revenue. The Hill curve has plateaued.

**Recommendation:** Reallocate budget away from Facebook toward Search, where marginal returns are highest. This is precisely the kind of actionable insight that pre-computed response curves make available to any analyst via a simple SQL query — no R or Python expertise required.

## 6. Model Registry Overview

Both the CausalImpact model and any Robyn models are tracked in the
same Snowflake Model Registry -- versioned, with metrics, governed.

In [ ]:
%%R
rcat("Registered models:")
sfr_show_models(reg)

In [ ]:
%%R
rcat("CausalImpact model versions:")
sfr_show_model_versions(reg, "CAUSAL_IMPACT_CAMPAIGN")

## 7. Cleanup

Uncomment to remove models and Feature Store resources.
Source data tables and schemas are managed by the setup notebook.

In [ ]:
# Cleanup: uncomment blocks below as needed.

# -- Delete models --
# from snowflake.snowpark.context import get_active_session
# _sess = get_active_session()
# _sess.sql("DROP MODEL IF EXISTS MARKETING_ANALYTICS_ML.MODELS.CAUSAL_IMPACT_CAMPAIGN").collect()
# print("Models deleted.")

print("Cleanup section -- uncomment blocks above to delete resources")

## Summary

This notebook demonstrated a complete marketing analytics workflow in
Snowflake with **schema-level isolation** for each concern:

| Schema | What we did |
|---|---|
| `RAW_DATA` | Queried source marketing data |
| `FEATURES` | Created 4 Feature Views (weekly spend pivot, weekly revenue, daily market OBJECT_AGG, CausalImpact market columns) |
| `TRAINING` | Generated Robyn training data from Feature Store joins |
| `MODELS` | Registered CausalImpact model with metrics |

### Key Capabilities Demonstrated

| Capability | How |
|---|---|
| R packages in Snowflake | CausalImpact and Robyn running natively via rpy2 |
| Feature Store | Reusable data pipelines: aggregation, PIVOT, OBJECT_AGG |
| Semi-structured data | OBJECT_AGG for flexible market-level analysis |
| Model Registry | Version control, metrics, governance for R models |
| Cross-language inference | R model retrieved from MR and scored in Python |
| SPCS serving (reference) | Production deployment pattern for R, Python, SQL access |

### Next Steps

- Connect to real marketing data sources in Snowflake
- Add more media channels to the Robyn model
- Implement Robyn response curve serving (Option A or B)
- Schedule periodic model retraining with Snowflake Tasks
- Explore model monitoring with Snowflake ML Observability

In [ ]:
# CI results -- writes a results table when run non-interactively
try:
    from snowflake.snowpark.context import get_active_session
    _ci_session = get_active_session()
    _ci_session.sql("""
        CREATE OR REPLACE TABLE NOTEBOOK_CI.MARKETING_DEMO_RESULTS AS
        SELECT 'marketing_demo' AS TEST_NAME, 'PASS' AS STATUS,
               CURRENT_TIMESTAMP() AS RUN_AT, CURRENT_USER() AS RUN_BY
    """).collect()
    print("CI results written to NOTEBOOK_CI.MARKETING_DEMO_RESULTS")
except Exception:
    pass